# 02 — Token Length Analysis

Compare tokenization characteristics across models:
- **TinyLlama 1.1B** (English-focused Llama tokenizer)
- **Qwen2.5 1.5B** (Multilingual — good for Bahasa Indonesia)

Key questions:
1. How many tokens does each domain produce on average?
2. What percentage of samples exceed the max_length (512 tokens)?
3. How does Qwen2.5 tokenize Bahasa Indonesia vs TinyLlama?

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer

from llm_finetuning.data.synthetic_generator import SyntheticGeneratorConfig, generate_dataset

In [ ]:
# Load models — uses cached versions if already downloaded
print('Loading tokenizers...')
tokenizers = {
    'TinyLlama-1.1B': AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0'),
    'Qwen2.5-1.5B': AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct'),
}
print('Tokenizers loaded.')

In [ ]:
samples = generate_dataset(SyntheticGeneratorConfig(num_samples=1000, seed=42))
MAX_LENGTH = 512

results = []
for s in samples:
    row = {'domain': s['domain']}
    for name, tok in tokenizers.items():
        try:
            text = tok.apply_chat_template(s['messages'], tokenize=False, add_generation_prompt=False)
            ids = tok(text, return_tensors=None, add_special_tokens=False)['input_ids']
            row[f'{name}_tokens'] = len(ids)
            row[f'{name}_exceeds'] = len(ids) > MAX_LENGTH
        except Exception:
            # Skip if model not downloaded
            row[f'{name}_tokens'] = None
            row[f'{name}_exceeds'] = None
    results.append(row)

df = pd.DataFrame(results)
df.head()

In [ ]:
# Token length comparison
for name in tokenizers:
    col = f'{name}_tokens'
    if df[col].notna().any():
        exceed_pct = df[f'{name}_exceeds'].mean() * 100
        print(f'{name}:')
        print(f'  Mean tokens: {df[col].mean():.1f}')
        print(f'  Median tokens: {df[col].median():.1f}')
        print(f'  Max tokens: {df[col].max()}')
        print(f'  Exceeds {MAX_LENGTH}: {exceed_pct:.1f}%')
        print()

In [ ]:
# Bahasa Indonesia tokenization comparison
id_samples = [
    'Machine learning adalah bagian dari kecerdasan buatan.',
    'Fine-tuning model bahasa besar membutuhkan sumber daya komputasi yang signifikan.',
    'Mekanisme atensi memungkinkan model untuk fokus pada bagian input yang relevan.',
]

print('Bahasa Indonesia tokenization comparison:')
print('=' * 60)
for text in id_samples:
    print(f'\nText: {text}')
    for name, tok in tokenizers.items():
        try:
            tokens = tok.tokenize(text)
            print(f'  {name} ({len(tokens)} tokens): {tokens[:10]}...' if len(tokens) > 10 else f'  {name} ({len(tokens)} tokens): {tokens}')
        except Exception as e:
            print(f'  {name}: Not available ({e})')